# Импорт библиотек

In [2]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(998, 213)

# Очистка таргета от выбросов

In [6]:
df = df[df['SI'] < 2000]

# Подготовка данных для эксперемента

In [8]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

In [9]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['SI'].apply(lambda v: 1 if v >= df['SI'].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (794, 100), (794,)
Test dataset size: (199, 100), (199,)


# Эксперемент с моделями

In [11]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
0,Logistic Regression,0.64,0.64,0.64,0.64,199.0,0.68
3,Random Forest,0.64,0.64,0.64,0.64,199.0,0.69
7,CatBoost,0.63,0.63,0.63,0.63,199.0,0.67
9,AdaBoost,0.63,0.63,0.63,0.63,199.0,0.68
5,Gradient Boosting,0.62,0.62,0.63,0.62,199.0,0.67
8,HistGradientBoosting,0.62,0.63,0.64,0.63,199.0,0.70
1,Decision Tree,0.61,0.61,0.61,0.61,199.0,0.63
4,XGBoost,0.61,0.61,0.62,0.61,199.0,0.67
6,LightGBM,0.58,0.58,0.59,0.58,199.0,0.67
2,KNeighbors,0.51,0.53,0.54,0.53,199.0,0.59


# Подбор гиперпараметров

In [26]:
param_dist = {
    'n_estimators': randint(50, 250),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

Лучшие параметры: {'max_depth': 11, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 107}


In [24]:
model = RandomForestClassifier(
    max_depth=11,
    max_features='log2',
    min_samples_leaf=5,
    min_samples_split=2,
    n_estimators=107,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.66
ROC AUC Score: 0.71


,precision,recall,f1-score,support
0,0.635514,0.701031,0.666667,97.000000
1,0.684783,0.617647,0.649485,102.000000
accuracy,0.658291,0.658291,0.658291,0.658291
macro avg,0.660148,0.659339,0.658076,199.000000
weighted avg,0.660767,0.658291,0.657860,199.000000
